# 02 – Data acquisition, wrangling and MySQL export
BSMM.8730 – Charging the Gap. Run the cells top to bottom in Google Colab.

## Step 1 – Set up

In [ ]:
# STEP 1 – Install / import libraries (Colab already has most of these)
!pip -q install beautifulsoup4 lxml
import os, re, io, json, glob, time
import requests
import pandas as pd
from bs4 import BeautifulSoup
from google.colab import files
HEADERS = {"User-Agent": "Mozilla/5.0 (UWindsor BSMM8730 student project)"}
os.makedirs("raw", exist_ok=True); os.makedirs("out", exist_ok=True)
def read_any(path):
    """Read a .csv or .xlsx/.xls file into a DataFrame."""
    if path.lower().endswith((".xlsx", ".xls")):
        return pd.read_excel(path)
    return pd.read_csv(path, encoding="latin-1", low_memory=False)
def find_file(*keys):
    """Find an uploaded file in raw/ whose name contains all keys."""
    for f in sorted(os.listdir("raw")):
        if all(k in f.lower() for k in keys): return os.path.join("raw", f)
    return None
print("Ready")

## Step 2 – Upload raw files (.csv or .xlsx)
Select all at once: your charging-stations file (name contains **station**), every `ev_by_fsa_<year>_Q<n>` file, the StatCan registrations file (name contains **20100025** or **registration**) and the StatCan population file (name contains **population**).

In [ ]:
# STEP 2 – Upload your raw files (select ALL at once):
#   your charging-stations Excel/CSV (name must contain "station"),
#   every ev_by_fsa_<year>_Q<n> file (.csv or .xlsx),
#   the StatCan registrations file (name contains "20100025" or "registration"),
#   the StatCan population file (name contains "population")
up = files.upload()
for name, data in up.items():
    open(os.path.join("raw", name), "wb").write(data)
print(sorted(os.listdir("raw")))

## Step 3 – Web scraping #1: Ontario Data Catalogue
Scrapes the dataset page with BeautifulSoup and collects the CSV link for every quarter (saved to `scraped_ev_csv_links.csv`).

In [ ]:
# STEP 3 – WEB SCRAPING #1 (Ontario Data Catalogue page -> every quarterly EV CSV link)
CATALOGUE = "https://data.ontario.ca/dataset/electric-vehicles-in-ontario-by-forward-sortation-area"
html = requests.get(CATALOGUE, headers=HEADERS, timeout=60).text
soup = BeautifulSoup(html, "lxml")
resource_pages = sorted({a["href"] for a in soup.find_all("a", href=True) if "/resource/" in a["href"]})
csv_links = []
for rp in resource_pages:
    url = rp if rp.startswith("http") else "https://data.ontario.ca" + rp
    if url.lower().endswith(".csv"):
        csv_links.append(url); continue
    try:
        s2 = BeautifulSoup(requests.get(url, headers=HEADERS, timeout=60).text, "lxml")
        title = s2.find("h1").get_text(" ", strip=True) if s2.find("h1") else ""
        for a in s2.find_all("a", href=True):
            if a["href"].lower().endswith(".csv"):
                csv_links.append((title, a["href"]))
                break
    except Exception as e:
        print("skip", url, e)
    time.sleep(0.5)   # be polite to the server
csv_links = list(dict.fromkeys(csv_links))
print(len(csv_links), "CSV links scraped")
for t in csv_links[:10]: print(t)
pd.DataFrame(csv_links, columns=["quarter_title", "csv_url"]).to_csv("out/scraped_ev_csv_links.csv", index=False)
# Optional: download any quarter you do not already have
# for title, link in csv_links:
#     q = re.search(r"Q([1-4])\s*(\d{4})", title)
#     if q:
#         fn = f"raw/ev_by_fsa_{q.group(2)}_Q{q.group(1)}.csv"
#         if not os.path.exists(fn):
#             open(fn, "wb").write(requests.get(link, headers=HEADERS, timeout=60).content)

## Step 4 – Semi-structured data: charging stations JSON (API)
Paste your API key into `API_KEY` to download the JSON directly (recommended – this is the semi-structured source). If left empty, your uploaded stations Excel/CSV is used.

In [ ]:
# STEP 4 – SEMI-STRUCTURED: charging stations as JSON from the NRCan/NLR API
# Paste your API key below. The JSON is downloaded fresh and saved in raw/ (this is the semi-structured source).
# If you leave API_KEY empty, the notebook uses your uploaded stations Excel/CSV instead.
API_KEY = ""          # <-- paste your key between the quotes
if API_KEY:
    url = ("https://developer.nlr.gov/api/alt-fuel-stations/v1.json"
           f"?api_key={API_KEY}&country=CA&state=ON&fuel_type=ELEC&status=all&access=all&limit=all")
    j = requests.get(url, timeout=120).json()
    json.dump(j, open("raw/ontario_ev_stations.json", "w", encoding="utf-8"))
    print("API total_results:", j.get("total_results"))
    st = pd.json_normalize(j["fuel_stations"])
else:
    path = find_file("station")
    print("Using uploaded file:", path)
    st = read_any(path)
# accept JSON field names, the API's own CSV/Excel headers, or our earlier cleaned file
st.columns = [str(c).strip() for c in st.columns]
aliases = {
 "station_id":["id","ID","station_id"], "station_name":["station_name","Station Name"],
 "street_address":["street_address","Street Address"], "city":["city","City"],
 "postal_code":["zip","ZIP","postal_code"], "latitude":["latitude","Latitude"],
 "longitude":["longitude","Longitude"], "network":["ev_network","EV Network","network"],
 "connector_types":["ev_connector_types","EV Connector Types","connector_types"],
 "level1_ports":["ev_level1_evse_num","EV Level1 EVSE Num","level1_ports"],
 "level2_ports":["ev_level2_evse_num","EV Level2 EVSE Num","level2_ports"],
 "dc_fast_ports":["ev_dc_fast_num","EV DC Fast Count","dc_fast_ports"],
 "access":["access_code","Access Code","access"], "status":["status_code","Status Code","status"],
 "open_date":["open_date","Open Date"]}
cols = {}
for new, opts in aliases.items():
    hit = next((o for o in opts if o in st.columns), None)
    cols[new] = st[hit] if hit else None
st = pd.DataFrame(cols)
print("Columns not found:", [k for k, v in cols.items() if v is None])
st["connector_types"] = st["connector_types"].apply(lambda x: ", ".join(x) if isinstance(x, list) else x)
for c in ["level1_ports","level2_ports","dc_fast_ports"]:
    st[c] = pd.to_numeric(st[c], errors="coerce").fillna(0).astype(int)
st["total_ports"] = st[["level1_ports","level2_ports","dc_fast_ports"]].sum(axis=1)
st["station_id"] = pd.to_numeric(st["station_id"], errors="coerce")
st = st.dropna(subset=["station_id"]); st["station_id"] = st["station_id"].astype(int)
st["city"] = st["city"].astype("string").str.strip().str.title()
st["postal_code"] = st["postal_code"].fillna("").astype(str).str.upper().str.replace(" ", "", regex=False)
st["fsa_code"] = st["postal_code"].str[:3].where(st["postal_code"].str.match(r"^[A-Z]\d[A-Z]"))
st["access"] = st["access"].astype("string").str.lower()
st["status"] = st["status"].astype("string").str.upper().str[:1]
st["open_date"] = pd.to_datetime(st["open_date"], errors="coerce").dt.date
st = st.drop_duplicates("station_id")
print("Stations:", len(st), "| no valid FSA:", st["fsa_code"].isna().sum())
st.head()

## Step 5 – Structured data: EV registrations by FSA (all quarters → one table)

In [ ]:
# STEP 5 – STRUCTURED: EV registrations by FSA (append all quarters into one long table)
def find_col(cols, *keys):
    for c in cols:
        lc = c.lower()
        if any(k in lc for k in keys): return c
    return None
frames = []
for path in sorted(glob.glob("raw/ev_by_fsa_*.*")):
    m = re.search(r"(\d{4})_Q([1-4])", path)
    period = f"{m.group(1)}-Q{m.group(2)}" if m else os.path.basename(path)
    d = read_any(path)
    d.columns = [str(c).strip() for c in d.columns]
    fsa = find_col(d.columns, "fsa", "sortation")
    bev = find_col(d.columns, "bev", "battery")
    phev = find_col(d.columns, "phev", "hybrid")
    tot = find_col(d.columns, "total")
    print(os.path.basename(path), "->", fsa, bev, phev, tot)
    out = pd.DataFrame({"fsa_code": d[fsa].astype(str).str.strip().str.upper().str[:3],
                        "period": period})
    for name, col in [("bev", bev), ("phev", phev), ("total_ev", tot)]:
        out[name] = pd.to_numeric(d[col].astype(str).str.replace(",", ""), errors="coerce") if col else None
    if out["total_ev"].isna().all():
        out["total_ev"] = out[["bev","phev"]].sum(axis=1)
    frames.append(out)
ev = pd.concat(frames, ignore_index=True)
ev = ev[ev["fsa_code"].str.match(r"^[A-Z]\d[A-Z]$")].drop_duplicates(["fsa_code","period"])
print("EV rows:", len(ev), "| quarters:", ev["period"].nunique())
ev.head()

## Step 6 – Structured data: population by FSA and StatCan provincial registrations

In [ ]:
# STEP 6 – STRUCTURED: StatCan population by FSA + provincial registrations
pop_raw = read_any(find_file("population"))
geo = find_col(pop_raw.columns, "geo")
pop_col = next((c for c in pop_raw.columns if "population, 2021" in c.lower()), None)
if pop_col:   # wide format
    pop = pop_raw[[geo, pop_col]].rename(columns={geo:"fsa_code", pop_col:"population"})
else:         # long format (VALUE column)
    char = next(c for c in pop_raw.columns if "population and dwelling" in c.lower())
    pop = pop_raw[pop_raw[char].str.contains("Population, 2021", na=False)][[geo, "VALUE"]]
    pop.columns = ["fsa_code", "population"]
pop["fsa_code"] = pop["fsa_code"].astype(str).str.strip().str[:3]
pop = pop[pop["fsa_code"].str.match(r"^[KLMNP]\d[A-Z]$")]          # Ontario FSAs start with K, L, M, N, P
pop["population"] = pd.to_numeric(pop["population"], errors="coerce")
pop = pop.drop_duplicates("fsa_code")
print("Ontario FSAs with population:", len(pop))

reg = read_any(find_file("20100025") or find_file("registration"))
fuel = find_col(reg.columns, "fuel")
vtype = find_col(reg.columns, "vehicle type")
reg = reg[["REF_DATE", "GEO", fuel, vtype, "VALUE"]]
reg.columns = ["period", "geography", "fuel_type", "vehicle_type", "vehicles"]
print("Provincial registration rows:", len(reg))
reg.head()

## Step 7 – Web scraping #2 + unstructured text: Government of Canada news releases
If the search returns 0 links, paste release URLs into `MANUAL_URLS`.

In [ ]:
# STEP 7 – WEB SCRAPING #2 + UNSTRUCTURED TEXT: Government of Canada news releases on EV chargers in Ontario
SEARCH = ("https://www.canada.ca/en/news/advanced-news-search/news-results.html"
          "?typ=newsreleases&dprtmnt=naturalresourcescanada&q=electric+vehicle+chargers+Ontario")
links = []
for page in range(1, 4):                                   # first 3 result pages
    s = BeautifulSoup(requests.get(SEARCH + f"&idx={(page-1)*10}", headers=HEADERS, timeout=60).text, "lxml")
    for a in s.find_all("a", href=True):
        h = a["href"]
        if "/news/" in h and h.endswith(".html") and "advanced-news-search" not in h:
            links.append(h if h.startswith("http") else "https://www.canada.ca" + h)
    time.sleep(1)
links = list(dict.fromkeys(links))
# If the search page returns nothing, paste release URLs manually here:
MANUAL_URLS = []
links += MANUAL_URLS
print(len(links), "news release links")

ontario_places = sorted(set(st["city"].dropna()) | {"Toronto","Ottawa","Windsor","London","Hamilton"}, key=len, reverse=True)
money_re = re.compile(r"\$\s?([\d.,]+)\s*(million|billion)?", re.I)
charger_re = re.compile(r"([\d,]+)\s+(?:new\s+)?(?:public\s+)?(?:electric vehicle\s+|EV\s+)?(?:fast\s+|level\s*2\s+|level\s*3\s+)?(?:charg(?:ers|ing stations|ing ports))", re.I)
rows = []
for i, url in enumerate(links, 1):
    try:
        s = BeautifulSoup(requests.get(url, headers=HEADERS, timeout=60).text, "lxml")
        main = s.find("main") or s
        text = " ".join(p.get_text(" ", strip=True) for p in main.find_all("p"))
        if "Ontario" not in text: continue
        date = (s.find("time") or {}).get("datetime", None) if s.find("time") else None
        m = money_re.search(text)
        amount = None
        if m:
            amount = float(m.group(1).replace(",", "")) * {"million":1e6, "billion":1e9}.get((m.group(2) or "").lower(), 1)
        c = charger_re.search(text)
        place = next((p for p in ontario_places if re.search(rf"\b{re.escape(p)}\b", text)), None)
        ctype = "DC fast" if re.search(r"fast charg|level 3", text, re.I) else ("Level 2" if re.search(r"level 2", text, re.I) else None)
        rows.append({"announcement_id": i, "release_date": date, "municipality": place,
                     "chargers_announced": int(c.group(1).replace(",", "")) if c else None,
                     "charger_type": ctype, "funding_amount": amount,
                     "source_url": url, "raw_text": text[:5000]})
    except Exception as e:
        print("skip", url, e)
    time.sleep(1)
fund = pd.DataFrame(rows)
city_fsa = st.dropna(subset=["fsa_code"]).groupby("city")["fsa_code"].agg(lambda x: x.mode()[0])
if len(fund):
    fund["fsa_code"] = fund["municipality"].map(city_fsa)
print("Announcements extracted:", len(fund))
fund.drop(columns="raw_text", errors="ignore").head(10)

## Step 8 – Integrate, quality checks and gap metrics

In [ ]:
# STEP 8 – Integrate + quality checks + gap metrics
fsa = pop.copy()
fsa["region"] = fsa["fsa_code"].str[0].map({"K":"Eastern Ontario","L":"Central Ontario","M":"Toronto",
                                            "N":"Southwestern Ontario","P":"Northern Ontario"})
public = st[(st["access"] == "public") & (st["status"] == "E") & st["fsa_code"].notna()]
ports = public.groupby("fsa_code").agg(public_ports=("total_ports","sum"),
                                       dc_fast_ports=("dc_fast_ports","sum")).reset_index()
latest = ev["period"].max()
gap = (fsa.merge(ev[ev["period"] == latest], on="fsa_code", how="left")
          .merge(ports, on="fsa_code", how="left").fillna({"public_ports":0, "dc_fast_ports":0}))
gap["evs_per_port"] = gap["total_ev"] / gap["public_ports"].replace(0, pd.NA)
gap["ports_per_1000"] = gap["public_ports"] / gap["population"] * 1000
print("Latest quarter:", latest)
print("Stations whose FSA is not in the population table:", (~st["fsa_code"].dropna().isin(fsa["fsa_code"])).sum())
print("FSAs with EVs but zero public ports:", ((gap["total_ev"] > 0) & (gap["public_ports"] == 0)).sum())
gap.sort_values("evs_per_port", ascending=False).head(10)

## Step 9 – Build the MySQL dump (schema + data)

In [ ]:
# STEP 9 – Write the MySQL dump (schema + data) for MySQL Workbench > Data Import
def q(v):
    if v is None or (isinstance(v, float) and pd.isna(v)) or v is pd.NA or (not isinstance(v, str) and pd.isna(v)):
        return "NULL"
    if isinstance(v, (int, float)) and not isinstance(v, bool):
        return str(v)
    return "'" + str(v).replace("\\", "\\\\").replace("'", "''") + "'"
def inserts(table, df, cols, batch=500):
    out = []
    for i in range(0, len(df), batch):
        vals = ",\n".join("(" + ",".join(q(r[c]) for c in cols) + ")" for _, r in df.iloc[i:i+batch].iterrows())
        out.append(f"INSERT INTO {table} ({','.join(cols)}) VALUES\n{vals};")
    return "\n".join(out)

known = set(fsa["fsa_code"])
st_sql = st.copy();  st_sql.loc[~st_sql["fsa_code"].isin(known), "fsa_code"] = None
ev_sql = ev[ev["fsa_code"].isin(known)]
fund_sql = fund.copy() if len(fund) else pd.DataFrame(columns=["announcement_id","release_date","municipality","fsa_code","chargers_announced","charger_type","funding_amount","source_url","raw_text"])
if len(fund_sql): fund_sql.loc[~fund_sql["fsa_code"].isin(known), "fsa_code"] = None

ddl = """DROP DATABASE IF EXISTS ev_charging_gap;
CREATE DATABASE ev_charging_gap CHARACTER SET utf8mb4;
USE ev_charging_gap;
CREATE TABLE fsa (fsa_code CHAR(3) PRIMARY KEY, region VARCHAR(40), population INT);
CREATE TABLE ev_registrations (fsa_code CHAR(3), period VARCHAR(10), bev INT, phev INT, total_ev INT,
  PRIMARY KEY (fsa_code, period), FOREIGN KEY (fsa_code) REFERENCES fsa(fsa_code));
CREATE TABLE charging_stations (station_id INT PRIMARY KEY, fsa_code CHAR(3), station_name VARCHAR(255),
  city VARCHAR(100), postal_code VARCHAR(10), latitude DECIMAL(9,6), longitude DECIMAL(9,6), network VARCHAR(100),
  connector_types VARCHAR(255), level1_ports INT, level2_ports INT, dc_fast_ports INT, total_ports INT,
  access VARCHAR(20), status CHAR(1), open_date DATE, FOREIGN KEY (fsa_code) REFERENCES fsa(fsa_code));
CREATE TABLE vehicle_registrations_prov (id INT AUTO_INCREMENT PRIMARY KEY, period VARCHAR(10),
  geography VARCHAR(150), fuel_type VARCHAR(80), vehicle_type VARCHAR(80), vehicles INT);
CREATE TABLE funding_announcements (announcement_id INT PRIMARY KEY, release_date VARCHAR(30),
  municipality VARCHAR(100), fsa_code CHAR(3), chargers_announced INT, charger_type VARCHAR(20),
  funding_amount DECIMAL(15,2), source_url VARCHAR(500), raw_text TEXT,
  FOREIGN KEY (fsa_code) REFERENCES fsa(fsa_code));
"""
view = """CREATE VIEW v_fsa_gap AS
SELECT f.fsa_code, f.region, f.population, e.period, e.total_ev,
       COALESCE(SUM(s.total_ports),0) AS public_ports,
       e.total_ev / NULLIF(SUM(s.total_ports),0) AS evs_per_port,
       COALESCE(SUM(s.total_ports),0) / f.population * 1000 AS ports_per_1000
FROM fsa f
JOIN ev_registrations e ON e.fsa_code = f.fsa_code
LEFT JOIN charging_stations s ON s.fsa_code = f.fsa_code AND s.access = 'public' AND s.status = 'E'
GROUP BY f.fsa_code, f.region, f.population, e.period, e.total_ev;
"""
sql = "\n".join([ddl,
    inserts("fsa", fsa, ["fsa_code","region","population"]),
    inserts("ev_registrations", ev_sql, ["fsa_code","period","bev","phev","total_ev"]),
    inserts("charging_stations", st_sql, ["station_id","fsa_code","station_name","city","postal_code","latitude",
            "longitude","network","connector_types","level1_ports","level2_ports","dc_fast_ports","total_ports",
            "access","status","open_date"]),
    inserts("vehicle_registrations_prov", reg, ["period","geography","fuel_type","vehicle_type","vehicles"]),
    inserts("funding_announcements", fund_sql, ["announcement_id","release_date","municipality","fsa_code",
            "chargers_announced","charger_type","funding_amount","source_url","raw_text"]) if len(fund_sql) else "",
    view])
open("out/ev_charging_gap_dump.sql", "w", encoding="utf-8").write(sql)
for name, d in [("fsa",fsa),("ev_registrations",ev_sql),("charging_stations",st_sql),
                ("vehicle_registrations_prov",reg),("funding_announcements",fund_sql)]:
    d.to_csv(f"out/{name}.csv", index=False)
print("Dump size (MB):", round(os.path.getsize("out/ev_charging_gap_dump.sql")/1e6, 2))

## Step 10 – Download outputs

In [ ]:
# STEP 10 – Download everything as one zip
!cd out && zip -q -r ../ev_charging_gap_output.zip .
files.download("ev_charging_gap_output.zip")